# Weekly LHC Daily Movement Report

**Purpose:** Identify members whose LHC-tracked fields changed within the selected week.

**Data source:** BRONZE SQL (fully independent of Qlik/QVD)

**Output:** Excel file → `H:\Quality_Assurance\LHC Movement-snapshot_{start}_to_{end}.xlsx`

---

## Version History

| Version | File | Key Change |
|---|---|---|
| v1 | `10_LHC/LHC_Daily_Movement_Verify.ipynb` | Initial build. Baseline from Paragon QVD. Change flags as 1/0. |
| v2 | `10_LHC/LHC_Daily_Movement_Verify_v2.ipynb` | Fix: DWOC `<= date` → `< next_day` (full-day inclusive). Result: 22→23 rows. |
| v3 | `10_LHC/LHC_Daily_Movement_Verify_v3.ipynb` | Fix: Baseline moved from Paragon QVD → BRONZE SQL. Eliminates cross-source Age at Entry mismatch. |
| **v4 (current)** | `10_LHC/LHC_Daily_Movement_Verify_v4.ipynb` | Fix: PersonNameMap moved from Paragon QVD → BRONZE SQL. Zero QVD dependency. Added date filter + Excel export. |

---

## Usage

Only change **cell 2 (Configuration)**:
```python
BASELINE_DATE  = pd.Timestamp('YYYY-MM-DD')   # Monday of previous week (or max snapshot - 4 days)
SNAPSHOT_DATES = pd.bdate_range('YYYY-MM-DD', 'YYYY-MM-DD')   # Mon–Fri of reporting week
FILTER_START   = pd.Timestamp('YYYY-MM-DD')   # usually same as SNAPSHOT_DATES start
FILTER_END     = pd.Timestamp('YYYY-MM-DD')   # usually same as SNAPSHOT_DATES end
```
Then **Run All**.

In [ ]:
import pandas as pd
import numpy as np
import pyodbc

In [ ]:
# ── Configuration — only edit this cell ──────────────────────────────────────

BASELINE_DATE  = pd.Timestamp('2026-02-16')                  # baseline (previous week Mon, or max snapshot - 4 days)
SNAPSHOT_DATES = pd.bdate_range('2026-02-17', '2026-02-27')  # reporting week(s), weekdays only

FILTER_START   = pd.Timestamp('2026-02-23')                  # filter start for Excel export
FILTER_END     = pd.Timestamp('2026-02-27')                  # filter end for Excel export

OUTPUT_DIR     = r"H:\Quality_Assurance"                     # Excel output folder

In [ ]:
# ── SQL Connection ────────────────────────────────────────────────────────────

conn = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=prdsql05.westfund.com.au;'
    'DATABASE=BRONZE;'
    'Trusted_Connection=yes;'
)
print("Connection established.")

In [ ]:
# ── Reference Data ────────────────────────────────────────────────────────────

# PersonNameMap — from BRONZE SQL
baseline_ds = BASELINE_DATE.strftime('%Y%m%d')
person_name_map = pd.read_sql(f"""
    SELECT person_id                    AS [Person ID],
           first_name + ' ' + surname   AS [Person Name]
    FROM   dbo.person_{baseline_ds}
""", conn).drop_duplicates('Person ID')
print(f"PersonNameMap: {len(person_name_map):,} records")

# ClearanceStatus map
clearance_map = pd.DataFrame({
    'Clearance Status': ['B', 'C', 'H', 'I', 'N', 'P', 'R', 'W'],
    'Transfer Status':  ['Newborn', 'No Previous Cover', 'Westfund to Westfund',
                         'In Progress', 'Not Required', 'Pending', 'Received', 'New Recruit']
})
print(f"ClearanceStatusMap: {len(clearance_map)} records")

In [ ]:
# ── load_daily_snapshot() ─────────────────────────────────────────────────────

def load_daily_snapshot(snap_date):
    """Load one day's snapshot from BRONZE SQL tables. Returns None if tables are missing."""
    ds      = snap_date.strftime('%Y%m%d')
    dt_next = (snap_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    try:
        mem = pd.read_sql(f"""
            SELECT membership_id  AS [Membership ID],
                   memship_status AS [Membership Status],
                   member1_loading,
                   member2_loading
            FROM   dbo.memship_{ds}
            WHERE  memship_status = 'A'
        """, conn)

        pm = pd.read_sql(f"""
            SELECT person_id                 AS [Person ID],
                   membership_id             AS [Membership ID],
                   CAST(relationship AS INT) AS Relationship
            FROM   dbo.person_membership_{ds}
            WHERE  status_flag = 'A'
            AND    relationship < 4
        """, conn)

        df = mem.merge(pm, on='Membership ID', how='inner')

        def fmt_loading(row):
            rel = int(row['Relationship'])
            v = row['member1_loading'] if rel == 1 \
                else row['member2_loading'] if rel > 1 else 0
            v = float(v) if pd.notna(v) else 0
            return f"{int(v)}%" if v == int(v) else f"{v}%"

        df['Member Loading %'] = df.apply(fmt_loading, axis=1)
        df.drop(columns=['member1_loading', 'member2_loading', 'Relationship'], inplace=True)

        per = pd.read_sql(f"""
            SELECT person_id        AS [Person ID],
                   age_at_entry     AS [Age at Entry],
                   remove_lhc       AS [LHC Been Removed],
                   clearance_status AS [Clearance Status]
            FROM   dbo.person_{ds}
        """, conn)

        df = df.merge(per, on='Person ID', how='inner')

        dwoc = pd.read_sql(f"""
            SELECT pp.person_id        AS [Person ID],
                   pp.no_hospital_days AS [Days without Cover]
            FROM   dbo.person_previous_fund AS pp
            WHERE  pp.previous_fund_version = (
                       SELECT MAX(pf.previous_fund_version)
                       FROM   dbo.person_previous_fund AS pf
                       WHERE  pf.person_id = pp.person_id
                       AND    pf.create_datetime < '{dt_next}'
                   )
        """, conn)

        df = df.merge(dwoc, on='Person ID', how='left')
        df['SnapShot Date'] = pd.Timestamp(snap_date)

        return df[[
            'Person ID', 'Membership ID', 'SnapShot Date', 'Membership Status',
            'Age at Entry', 'Member Loading %', 'Days without Cover',
            'LHC Been Removed', 'Clearance Status'
        ]]

    except Exception as e:
        print(f"  SKIPPED ({e.__class__.__name__}: {e})")
        return None

In [ ]:
# ── Baseline ──────────────────────────────────────────────────────────────────

print(f"Loading baseline {BASELINE_DATE.date()} from BRONZE SQL ...", end=' ')
baseline = load_daily_snapshot(BASELINE_DATE)
print(f"{len(baseline):,} rows")

In [ ]:
# ── Load snapshots from BRONZE SQL ───────────────────────────────────────────

frames = []
for d in SNAPSHOT_DATES:
    print(f"Loading {d.date()} ...", end=' ')
    snap = load_daily_snapshot(d)
    if snap is not None:
        frames.append(snap)
        print(f"{len(snap):,} rows")

new_data = pd.concat(frames, ignore_index=True)
print(f"\nTotal new rows: {len(new_data):,}")

In [ ]:
# ── Calculate change flags ────────────────────────────────────────────────────

combined = (
    pd.concat([baseline, new_data], ignore_index=True)
    .sort_values(['Person ID', 'SnapShot Date'])
    .reset_index(drop=True)
)

combined['Days without Cover'] = pd.to_numeric(combined['Days without Cover'], errors='coerce')

# New Member Flag
combined['New Mbr Flag'] = (combined.groupby('Person ID').cumcount() == 0).astype(int)

# Change flags
for src_col, cur_col, prev_col, flag_col in [
    ('Age at Entry',      'Current CAE',             'Previous CAE',             'CAE Change Flag'),
    ('Days without Cover','Current DWOC',             'Previous DWOC',            'DWOC Change Flag'),
    ('Member Loading %',  'Current Member Loading %', 'Previous Member Loading %','Member Loading % Change Flag'),
]:
    prev    = combined.groupby('Person ID')[src_col].shift(1)
    changed = prev.notna() & (prev != combined[src_col])
    combined[cur_col]  = combined[src_col]
    combined[prev_col] = prev.where(changed)
    combined[flag_col] = np.where(changed, 'Yes', 'No')

# LHC Removed Flag
prev_lhc    = combined.groupby('Person ID')['LHC Been Removed'].shift(1)
lhc_changed = prev_lhc.notna() & (prev_lhc != combined['LHC Been Removed'])
combined['LHC Removed Flag'] = np.where(lhc_changed, 'Yes', 'No')

# Apply reference maps
combined = combined.merge(person_name_map, on='Person ID', how='left')
combined['Person Name']     = combined['Person Name'].fillna('Unknown Name')
combined = combined.merge(clearance_map, on='Clearance Status', how='left')
combined['Transfer Status'] = combined['Transfer Status'].fillna('No Transfer Status')

print("Change flags calculated.")

In [ ]:
# ── Final output — changed rows only ─────────────────────────────────────────

output = (
    combined[combined['SnapShot Date'].isin(SNAPSHOT_DATES)]
    .reset_index(drop=True)
    .copy()
)

output['Changed?'] = np.where(
    (output['CAE Change Flag'] == 'Yes') |
    (output['LHC Removed Flag'] == 'Yes') |
    (output['Member Loading % Change Flag'] == 'Yes') |
    (output['DWOC Change Flag'] == 'Yes'),
    'Changed', None
)

FINAL_COLS = [
    'Changed?',
    'Membership ID', 'Person ID', 'Membership Status', 'Person Name', 'SnapShot Date',
    'New Mbr Flag', 'Transfer Status',
    'Current CAE', 'Previous CAE', 'CAE Change Flag',
    'Previous DWOC', 'Current DWOC', 'DWOC Change Flag',
    'Previous Member Loading %', 'Current Member Loading %', 'Member Loading % Change Flag',
    'LHC Removed Flag'
]

final = output[FINAL_COLS][output['Changed?'] == 'Changed'].reset_index(drop=True)

print(f"Changed rows: {len(final):,} rows × {len(final.columns)} columns")
final

In [ ]:
# ── Filter by SnapShot Date range ─────────────────────────────────────────────

filtered = final[
    (final['SnapShot Date'] >= FILTER_START) &
    (final['SnapShot Date'] <= FILTER_END)
].reset_index(drop=True)

start_str = FILTER_START.strftime('%d%m%Y')
end_str   = FILTER_END.strftime('%d%m%Y')
var_name  = f"df_snapshot_{start_str}_to_{end_str}"

globals()[var_name] = filtered

print(f"SnapShot Date 范围：{FILTER_START.date()} 至 {FILTER_END.date()}")
print(f"筛选后行数：        {len(filtered):,}")
print(f"DataFrame 变量名：  {var_name}")
filtered

In [ ]:
# ── Export to Excel ───────────────────────────────────────────────────────────

file_name   = f"LHC Movement-snapshot_{start_str}_to_{end_str}.xlsx"
output_path = f"{OUTPUT_DIR}\\{file_name}"

filtered.to_excel(output_path, index=False, sheet_name='LHC Movement')

print(f"Saved: {output_path}")